## Setup

In [1]:
import subprocess, sys

def pip(args, desc=''):
    if desc:
        print(f'\n⏳ {desc}...')
    cmd = [sys.executable, '-m', 'pip'] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-3000:])
        raise RuntimeError(f"pip basarisiz: {' '.join(args[:3])}")
    print('OK')

# ── NEDEN NİGHTLY TORCH? ───────────────────────────────────────────────
# litert-torch-nightly artık torch >= 2.11.0 gerektiriyor.
# torch 2.11 henüz stable değil → nightly indeksinden almak gerekiyor.
# torchao da nightly alınmalı — aksi halde ABI uyumsuzluğu çıkıyor.
#
# Colab default CUDA: 12.8 → cu128
CUDA_VER = 'cu128'
NIGHTLY_INDEX = f'https://download.pytorch.org/whl/nightly/{CUDA_VER}'

# 0. Temizlik
pip(['uninstall', '-y', 'torch', 'torchvision', 'torchaudio', 'torchao',
     'litert-torch', 'litert-torch-nightly'],
    'Eski paketler kaldiriliyor')

# 1. torch nightly (>=2.11 için)
pip(['install',
     'torch', 'torchvision', 'torchaudio',
     '--index-url', NIGHTLY_INDEX,
     '--pre', '--quiet'],
    f'torch nightly kuruluyor ({CUDA_VER})')

# 2. torchao nightly — torch nightly ile ABI uyumlu
pip(['install', 'torchao',
     '--index-url', NIGHTLY_INDEX,
     '--pre', '--quiet'],
    'torchao nightly kuruluyor')

# 3. litert-torch-nightly
pip(['install', 'litert-torch-nightly', '--quiet'],
    'litert-torch-nightly kuruluyor')

# 4. litert-lm
pip(['install', 'litert-lm', '--quiet'],
    'litert-lm kuruluyor')

# 4.5. protobuf — litert-torch-nightly'nin gerektirdiği sürüm
pip(['install', 'protobuf>=6.31.1', '--quiet'],
    'protobuf yükseltiliyor')

# 5. HuggingFace
pip(['install', 'huggingface-hub', 'transformers>=4.50.0', 'accelerate', '--quiet'],
    'HuggingFace araclari kuruluyor')

print('\n' + '=' * 60)
print('KURULUM TAMAM')
print('>>> Runtime -> Restart session (Ctrl+M .) yapip devam et <<<')
print('=' * 60)


⏳ Eski paketler kaldiriliyor...
OK

⏳ torch nightly kuruluyor (cu128)...
OK

⏳ torchao nightly kuruluyor...
OK

⏳ litert-torch-nightly kuruluyor...
OK

⏳ litert-lm kuruluyor...
OK

⏳ protobuf yükseltiliyor...
OK

⏳ HuggingFace araclari kuruluyor...
OK

KURULUM TAMAM
>>> Runtime -> Restart session (Ctrl+M .) yapip devam et <<<


#### ---------------- Kernel Restart

In [1]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install',
    'git+https://github.com/huggingface/transformers.git',
    '--quiet'], check=True)
print('transformers source kuruldu')



transformers source kuruldu


In [2]:
import sys
import google.protobuf
import transformers


# ── ADIM 1: Versiyon kontrolü ─────────────────────────────────────────
import torch
import torchao
print(f'torch   : {torch.__version__}')
print(f'torchao : {torchao.__version__}')
print(f"protobuf : {google.protobuf.__version__}")
print(f'transformers : {transformers.__version__}')

# ── ADIM 2: Gerekli torchao alt modüllerini önceden yükle ─────────────
# litert_torch, import zincirinde bu modülleri arar.
# Eğer torchao'yu import etmeden önce litert_torch yüklenirse
# Python'un lazy import mekanizması bazı durumlarda başarısız oluyor.
# Çözüm: hepsini sırayla import ederek sys.modules'u doldur.
import torchao.quantization
import torchao.quantization.pt2e
import torchao.quantization.pt2e.quantize_pt2e
import torchao.quantization.pt2e.quantizer
print('torchao alt modulleri yuklendi')

# ── ADIM 3: litert_torch import ───────────────────────────────────────
try:
    import litert_torch
    print(f'litert_torch {litert_torch.__version__} hazir')
except Exception as e:
    print(f'HATA: {e}')
    import traceback; traceback.print_exc()
    print()
    print('Kontrol listesi:')
    print('  1. Hucre 1 calistirildi mi?')
    print('  2. Kernel yeniden baslatildi mi?')
    print('  3. Bu hucreyi tekrar calistir')

W0503 08:26:20.608000 3754 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


torch   : 2.12.0.dev20260407+cu128
torchao : 0.18.0.dev20260407+cu128
protobuf : 7.34.1
transformers : 5.8.0.dev0
torchao alt modulleri yuklendi
litert_torch 0.10.0.dev20260502 hazir


In [3]:
import os
from getpass import getpass

# ============================================================
# ⚙️  EXPORT AYARLARI
# ============================================================

# Model — orijinal HF modeli veya fine-tuned local path
# Örnekler:
#   "google/gemma-4-E2B-it"     → orijinal E2B
#   "google/gemma-4-E4B-it"     → orijinal E4B
#   "./merged_model"            → local fine-tuned (merge edilmiş safetensors)
MODEL_ID = "google/gemma-4-E2B-it"  # ← DEĞİŞTİR

# Çıktı dizini
OUTPUT_DIR = "/tmp/gemma4_litert_export"

# ── QUANTIZATION REÇETESİ ────────────────────────────────────
# Değer              | Açıklama                              | Tavsiye
# -------------------+---------------------------------------+---------
# dynamic_wi4_afp32  | INT4 ağırlık, FP32 aktivasyon (2-4-8 | ★ Önerilen (Google'ın kullandığı)
#                    | bit karışımı, küçük dosya)            |
# dynamic_wi8_afp32  | INT8 ağırlık, FP32 aktivasyon        | Daha hassas, daha büyük
# (boş string)       | Quantization yok, float32            | Üretim için uygun değil
#
# ⚠️ NOT: dynamic_wi8_afp32 ile de <pad> üretiyorsa büyük olasılıkla
#    --jinja_chat_template_override eksik ya da yanlış.
QUANTIZE = "dynamic_wi4_afp32"  # ← DEĞİŞTİR

# ── PREFILL (Prompt işleme chunk boyutu) ─────────────────────
# Değer  | Açıklama
# -------+------------------------------------------------------
# 128    | Çok kısıtlı RAM (≤4GB) veya eski telefon
# 256    | Mobil default (Google'ın benchmark ayarı)
# 512    | Dengeli (tablet / güçlü telefon)
# 1024   | Masaüstü / NPU varsa (en hızlı TTFT)
# 2048   | Sunucu / yüksek VRAM
#
# Büyük değer → ilk token daha hızlı ama RAM gereksinimi artar
PREFILL_SEQ_LEN = 512  # ← DEĞİŞTİR

# ── KV CACHE (Toplam context penceresi) ──────────────────────
# Değer  | Açıklama
# -------+------------------------------------------------------
# 1024   | Kısa sohbet (telefon, RAM kısıtlı)
# 2048   | Mobil standart (önerilen minimum)
# 4096   | Uzun bağlam, tablet
# 8192   | Masaüstü / yüksek RAM
#
# Gemma 4 E2B/E4B teorik max: 128K — ama LiteRT-LM'de bu kadar desteklenmiyor
CACHE_LENGTH = 2048  # ← DEĞİŞTİR

# ── MULTIMODAL ───────────────────────────────────────────────
# externalize_embedder=True → görsel/audio encoder ayrı tflite olarak export edilir
# Bu Gemma 4 E2B/E4B için ZORUNLU (PLE mimarisi)
# False yaparsanız model export edilir ama görsel inference çalışmaz
EXTERNALIZE_EMBEDDER = True  # E2B/E4B için her zaman True bırak

# ============================================================

# HF Token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("✅ HF_TOKEN Colab Secrets'tan alındı")
except Exception:
    HF_TOKEN = getpass("HuggingFace Token gir (gizli kalır): ")

os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Chat template — <pad> sorununun ANA ÇÖZÜMü budur!
# Bu flag olmadan model varsayılan template kullanır ve yanlış token üretir.
# litert-community'nin pre-export repolarındaki jinja template'i indirir.
if "E4B" in MODEL_ID:
    TEMPLATE_REPO = "litert-community/gemma-4-E4B-it-litert-lm"
else:
    TEMPLATE_REPO = "litert-community/gemma-4-E2B-it-litert-lm"

print("\n📋 Export Ayarları:")
print(f"  Model              : {MODEL_ID}")
print(f"  Çıktı              : {OUTPUT_DIR}")
print(f"  Quantize           : {QUANTIZE or '(yok — float32)'}")
print(f"  Prefill seq len    : {PREFILL_SEQ_LEN}")
print(f"  KV Cache length    : {CACHE_LENGTH}")
print(f"  Externalize embed  : {EXTERNALIZE_EMBEDDER}")
print(f"  Chat template      : {TEMPLATE_REPO}")
print()
print("💡 <pad> sorununa karşı kontrol listesi:")
print(f"   [{'✓' if EXTERNALIZE_EMBEDDER else '✗'}] externalize_embedder = True")
print(f"   [✓] jinja_chat_template_override = {TEMPLATE_REPO}")
print(f"   [{'✓' if QUANTIZE == 'dynamic_wi4_afp32' else '!'}] quantize = {QUANTIZE or 'float32 (risk var)'}")

✅ HF_TOKEN Colab Secrets'tan alındı


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.



📋 Export Ayarları:
  Model              : google/gemma-4-E2B-it
  Çıktı              : /tmp/gemma4_litert_export
  Quantize           : dynamic_wi4_afp32
  Prefill seq len    : 512
  KV Cache length    : 2048
  Externalize embed  : True
  Chat template      : litert-community/gemma-4-E2B-it-litert-lm

💡 <pad> sorununa karşı kontrol listesi:
   [✓] externalize_embedder = True
   [✓] jinja_chat_template_override = litert-community/gemma-4-E2B-it-litert-lm
   [✓] quantize = dynamic_wi4_afp32


In [6]:
from huggingface_hub import hf_hub_download

# tokenizer_config.json yok, doğrudan jinja dosyasını indir
path = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="chat_template.jinja"
)

with open(path) as f:
    template = f.read()

print(template[:500])  # ilk 500 karakter

chat_template.jinja: 0.00B [00:00, ?B/s]

{%- macro format_parameters(properties, required) -%}
    {%- set standard_keys = ['description', 'type', 'properties', 'required', 'nullable'] -%}
    {%- set ns = namespace(found_first=false) -%}
    {%- for key, value in properties | dictsort -%}
        {%- set add_comma = false -%}
        {%- if key not in standard_keys -%}
            {%- if ns.found_first %},{% endif -%}
            {%- set ns.found_first = true -%}
            {{ key }}:{
            {%- if value['description'] -%}
    


In [7]:
import subprocess, time, os

# ── Export komutunu oluştur ────────────────────────────────────────────
cmd_parts = [
    "litert-torch export_hf",
    MODEL_ID,
    OUTPUT_DIR,
    "--task=text_generation",          # ← buraya
]

if EXTERNALIZE_EMBEDDER:
    cmd_parts.append("--externalize_embedder=True")

cmd_parts.append(f"--jinja_chat_template_override={TEMPLATE_REPO}")

if QUANTIZE:
    cmd_parts.append(f"--quantization_recipe={QUANTIZE}")

cmd_parts.append(f"-p {PREFILL_SEQ_LEN}")
cmd_parts.append(f"--cache_length {CACHE_LENGTH}")

cmd = " \\\n  ".join(cmd_parts)

print("📤 Çalıştırılacak komut:")
print("-" * 60)
print(cmd)
print("-" * 60)
print(f"\n🚀 Export başlıyor... (E2B ~30dk, E4B ~60dk bekleniyor)")

start_time = time.time()
env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN

process = subprocess.Popen(
    cmd.replace(" \\\n  ", " "),
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env
)

output_lines = []
for line in process.stdout:
    line = line.rstrip()
    output_lines.append(line)
    if any(kw in line.lower() for kw in
           ['error', 'warning', 'export', 'convert', 'quant', 'saving',
            'loading', 'done', 'complet', 'write', 'model', '%', 'step',
            'embedder', 'template', 'jinja', 'lm_head']):
        print(line)

process.wait()
elapsed = time.time() - start_time

print("\n" + "=" * 60)
if process.returncode == 0:
    print(f"✅ EXPORT BAŞARILI — {elapsed/60:.1f} dk sürdü")
    print(f"\n📁 {OUTPUT_DIR} içeriği:")
    for root, dirs, files in os.walk(OUTPUT_DIR):
        level = root.replace(OUTPUT_DIR, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files:
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath) / 1e6
            print(f"{'  ' * (level+1)}{f}  ({size:.1f} MB)")
else:
    print(f"❌ EXPORT BAŞARISIZ — Return code: {process.returncode}")
    print("\nSon 40 satır çıktı:")
    for line in output_lines[-40:]:
        print(line)
print("=" * 60)

📤 Çalıştırılacak komut:
------------------------------------------------------------
litert-torch export_hf \
  google/gemma-4-E2B-it \
  /tmp/gemma4_litert_export \
  --task=text_generation \
  --externalize_embedder=True \
  --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm \
  --quantization_recipe=dynamic_wi4_afp32 \
  -p 512 \
  --cache_length 2048
------------------------------------------------------------

🚀 Export başlıyor... (E2B ~30dk, E4B ~60dk bekleniyor)
I0000 00:00:1777796872.741059    4463 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0503 08:27:54.599000 4463 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is 

In [ ]:
import subprocess, os

LITERTLM_PATH = "/tmp/gemma4_litert_export/model.litertlm"

# ── TEST 1: Sadece text ───────────────────────────────────────────────
print("=" * 60)
print("TEST 1 — Text only")
print("=" * 60)

TEXT_PROMPT = "Who are you?"

result = subprocess.run(
    f'litert-lm run "{LITERTLM_PATH}" --prompt="{TEXT_PROMPT}"',
    shell=True, capture_output=True, text=True, timeout=180
)

if result.stdout:
    print(result.stdout)
if result.returncode != 0:
    print(f"❌ Hata (kod {result.returncode}):")
    print(result.stderr[-1000:])
else:
    pad_count = result.stdout.count('<pad>')
    if pad_count > 3:
        print(f"⚠️  <pad> sorunu ({pad_count} adet)")
    else:
        print("✅ Text inference başarılı!")



TEST 1 — Text only
I am Gemma 4, a Large Language Model developed by Google DeepMind. I am an open weights model.

✅ Text inference başarılı!


In [9]:
from huggingface_hub import HfApi

# Ayarlar
repo_id = "SalihHub/gemma-4-E2B-it-litert_try"  # Burayı kendi kullanıcı adınla değiştir
file_path = "/tmp/gemma4_litert_export/model.litertlm"
path_in_repo = "model.litertlm"

api = HfApi()

# Eğer repo henüz yoksa oluştur (Opsiyonel)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

print(f"🚀 Yükleme başlıyor: {repo_id}")

api.upload_file(
    path_or_fileobj=file_path,
    path_in_repo=path_in_repo,
    repo_id=repo_id,
    repo_type="model",
)

print(f"✅ Başarıyla yüklendi: https://huggingface.co/{repo_id}")

🚀 Yükleme başlıyor: SalihHub/gemma-4-E2B-it-litert_try


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ert_export/model.litertlm:   1%|          | 17.4MB / 2.56GB            

✅ Başarıyla yüklendi: https://huggingface.co/SalihHub/gemma-4-E2B-it-litert_try


In [13]:
from huggingface_hub import hf_hub_download

# Resmi modeli indir (cache'deyse tekrar indirmez)
print("Resmi model indiriliyor...")
OFFICIAL_PATH = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="gemma-4-E2B-it.litertlm"
)
print(f"✅ OFFICIAL_PATH = {OFFICIAL_PATH}")
print(f"MY_PATH = {MY_PATH}")

Resmi model indiriliyor...


gemma-4-E2B-it.litertlm:   0%|          | 0.00/2.58G [00:00<?, ?B/s]

✅ OFFICIAL_PATH = /root/.cache/huggingface/hub/models--litert-community--gemma-4-E2B-it-litert-lm/snapshots/84b6978eff6e4eea02825bc2ee4ea48579f13109/gemma-4-E2B-it.litertlm
MY_PATH = /root/.cache/huggingface/hub/models--SalihHub--gemma-4-E2B-it-litert_try/snapshots/b26b97fd5de39ee304068193c338da373e73f282/model.litertlm


In [14]:
import struct
from huggingface_hub import hf_hub_download

OFFICIAL_PATH = "/root/.cache/huggingface/hub/models--litert-community--gemma-4-E2B-it-litert-lm/blobs/ab7838cdfc8f77e54d8ca45eadceb20452d9f01e4bfade03e5dce27911b27e42"
MY_PATH = hf_hub_download(repo_id="SalihHub/gemma-4-E2B-it-litert_try", filename="model.litertlm")

SECTION_TYPES = {
    0: "NONE", 1: "GenericBinaryData", 2: "Deprecated",
    3: "TFLiteModel", 4: "SP_Tokenizer", 5: "LlmMetadataProto",
    6: "HF_Tokenizer_Zlib", 7: "TFLiteWeights",
}

def read_string(buf, ptr_pos):
    rel = struct.unpack_from('<i', buf, ptr_pos)[0]
    sp = ptr_pos + rel
    ln = struct.unpack_from('<I', buf, sp)[0]
    return buf[sp+4:sp+4+ln].decode('utf-8', 'replace')

def read_kv(buf, kv_pos):
    vt_off = struct.unpack_from('<i', buf, kv_pos)[0]
    vt_pos = kv_pos - vt_off
    vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
    num_fields = (vt_size - 4) // 2

    def get_field(fi):
        if fi >= num_fields: return None
        off = struct.unpack_from('<H', buf, vt_pos + 4 + fi*2)[0]
        return (kv_pos + off) if off else None

    key_p = get_field(0)
    key = read_string(buf, key_p) if key_p else ''

    val_table_ptr = get_field(2)
    val = ''
    if val_table_ptr:
        val_table_rel = struct.unpack_from('<i', buf, val_table_ptr)[0]
        val_table_pos = val_table_ptr + val_table_rel
        vt2_off = struct.unpack_from('<i', buf, val_table_pos)[0]
        vt2_pos = val_table_pos - vt2_off
        str_off = struct.unpack_from('<H', buf, vt2_pos + 4)[0]
        if str_off:
            val = read_string(buf, val_table_pos + str_off)
    return key, val

def parse_litertlm_with_values(path):
    with open(path, 'rb') as f:
        f.read(8); f.read(12); f.read(4)
        header_end_offset = struct.unpack('<Q', f.read(8))[0]
        fb_data = f.read(header_end_offset - f.tell())
    buf = bytearray(fb_data)

    def rtf(buf, table_pos, fi):
        vt_off = struct.unpack_from('<i', buf, table_pos)[0]
        vt_pos = table_pos - vt_off
        vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
        fs = 4 + fi * 2
        if fs + 2 > vt_size: return None
        fo = struct.unpack_from('<H', buf, vt_pos + fs)[0]
        return (table_pos + fo) if fo else None

    root_pos = struct.unpack_from('<I', buf, 0)[0]
    sm_ptr = rtf(buf, root_pos, 1)
    sm_pos = sm_ptr + struct.unpack_from('<i', buf, sm_ptr)[0]
    obj_ptr = rtf(buf, sm_pos, 0)
    vec_pos = obj_ptr + struct.unpack_from('<i', buf, obj_ptr)[0]
    num_objects = struct.unpack_from('<I', buf, vec_pos)[0]

    results = []
    for i in range(num_objects):
        elem_ptr = vec_pos + 4 + i * 4
        obj_pos = elem_ptr + struct.unpack_from('<i', buf, elem_ptr)[0]
        p = rtf(buf, obj_pos, 1)
        begin_off = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(buf, obj_pos, 2)
        end_off = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(buf, obj_pos, 3)
        data_type = struct.unpack_from('<B', buf, p)[0] if p else 0

        kv_pairs = []
        items_ptr = rtf(buf, obj_pos, 0)
        if items_ptr:
            items_vec = items_ptr + struct.unpack_from('<i', buf, items_ptr)[0]
            num_items = struct.unpack_from('<I', buf, items_vec)[0]
            for j in range(num_items):
                kv_ptr = items_vec + 4 + j * 4
                kv_pos = kv_ptr + struct.unpack_from('<i', buf, kv_ptr)[0]
                key, val = read_kv(buf, kv_pos)
                kv_pairs.append((key, val))

        size_mb = (end_off - begin_off) / 1e6
        type_name = SECTION_TYPES.get(data_type, f"unknown({data_type})")
        results.append({'idx': i, 'data_type': data_type, 'type': type_name,
                        'begin': begin_off, 'end': end_off, 'labels': kv_pairs})
    return results

# ── ÇALIŞTIR ──────────────────────────────────────────────────────────
print("=" * 70)
print("RESMİ MODEL")
print("=" * 70)
official_sections = parse_litertlm_with_values(OFFICIAL_PATH)
for s in official_sections:
    size = (s['end'] - s['begin']) / 1e6
    print(f"[{s['idx']:>2}] {s['type']:<22} {size:>8.1f} MB  {s['labels']}")

print("\n" + "=" * 70)
print("KENDİ MODELİN")
print("=" * 70)
my_sections = parse_litertlm_with_values(MY_PATH)
for s in my_sections:
    size = (s['end'] - s['begin']) / 1e6
    print(f"[{s['idx']:>2}] {s['type']:<22} {size:>8.1f} MB  {s['labels']}")

RESMİ MODEL
[ 0] LlmMetadataProto            0.0 MB  []
[ 1] SP_Tokenizer                4.7 MB  []
[ 2] TFLiteModel               103.8 MB  [('model_type', 'tf_lite_embedder')]
[ 3] TFLiteModel              1284.5 MB  [('model_type', 'tf_lite_per_layer_embedder')]
[ 4] TFLiteModel                94.1 MB  [('model_type', 'tf_lite_audio_encoder_hw'), ('backend_constraint', 'cpu')]
[ 5] TFLiteModel                 9.4 MB  [('model_type', 'tf_lite_audio_adapter'), ('backend_constraint', 'cpu')]
[ 6] TFLiteModel                 0.0 MB  [('model_type', 'tf_lite_end_of_audio')]
[ 7] TFLiteModel               219.1 MB  [('model_type', 'tf_lite_vision_encoder')]
[ 8] TFLiteModel                 4.7 MB  [('model_type', 'tf_lite_vision_adapter'), ('backend_constraint', 'cpu')]
[ 9] TFLiteModel                 0.0 MB  [('model_type', 'tf_lite_end_of_vision')]
[10] TFLiteModel               818.3 MB  [('model_type', 'tf_lite_prefill_decode')]
[11] TFLiteModel                44.3 MB  [('model_type'

In [15]:
import os
path = "/tmp/gemma4_litert_export/model.litertlm"
if os.path.exists(path):
    size = os.path.getsize(path) / 1e9
    print(f"✅ Export hazır: {size:.2f} GB")
else:
    print("❌ Export yok — önce export hücresini çalıştır")

✅ Export hazır: 2.56 GB


In [21]:
##Merge models

import struct, flatbuffers
from huggingface_hub import hf_hub_download, HfApi

# ── PATH'LER ─────────────────────────────────────────────────────────
MY_PATH       = "/tmp/gemma4_litert_export/model.litertlm"
OFFICIAL_PATH = hf_hub_download(
    repo_id="litert-community/gemma-4-E2B-it-litert-lm",
    filename="gemma-4-E2B-it.litertlm"
)
OUTPUT_PATH   = "/tmp/merged_final.litertlm"
BLOCK_SIZE    = 16 * 1024

SECTION_TYPES = {
    0:"NONE", 1:"GenericBinaryData", 2:"Deprecated",
    3:"TFLiteModel", 4:"SP_Tokenizer", 5:"LlmMetadataProto", 6:"HF_Tokenizer_Zlib", 7:"TFLiteWeights",
}

# ── PARSER ───────────────────────────────────────────────────────────
def read_string(buf, ptr_pos):
    rel = struct.unpack_from('<i', buf, ptr_pos)[0]
    sp = ptr_pos + rel
    ln = struct.unpack_from('<I', buf, sp)[0]
    return buf[sp+4:sp+4+ln].decode('utf-8', 'replace')

def read_kv(buf, kv_pos):
    vt_off = struct.unpack_from('<i', buf, kv_pos)[0]
    vt_pos = kv_pos - vt_off
    vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
    num_fields = (vt_size - 4) // 2
    def get_field(fi):
        if fi >= num_fields: return None
        off = struct.unpack_from('<H', buf, vt_pos + 4 + fi*2)[0]
        return (kv_pos + off) if off else None
    key_p = get_field(0)
    key = read_string(buf, key_p) if key_p else ''
    val_table_ptr = get_field(2)
    val = ''
    if val_table_ptr:
        vtp = val_table_ptr + struct.unpack_from('<i', buf, val_table_ptr)[0]
        vt2_pos = vtp - struct.unpack_from('<i', buf, vtp)[0]
        str_off = struct.unpack_from('<H', buf, vt2_pos + 4)[0]
        if str_off:
            val = read_string(buf, vtp + str_off)
    return key, val

def parse_litertlm(path):
    with open(path, 'rb') as f:
        f.read(8); f.read(12); f.read(4)
        heo = struct.unpack('<Q', f.read(8))[0]
        fb_data = f.read(heo - f.tell())
    buf = bytearray(fb_data)
    def rtf(tp, fi):
        vt_pos = tp - struct.unpack_from('<i', buf, tp)[0]
        vt_size = struct.unpack_from('<H', buf, vt_pos)[0]
        fs = 4 + fi * 2
        if fs + 2 > vt_size: return None
        fo = struct.unpack_from('<H', buf, vt_pos + fs)[0]
        return (tp + fo) if fo else None
    root_pos = struct.unpack_from('<I', buf, 0)[0]
    sm_ptr = rtf(root_pos, 1)
    sm_pos = sm_ptr + struct.unpack_from('<i', buf, sm_ptr)[0]
    obj_ptr = rtf(sm_pos, 0)
    vec_pos = obj_ptr + struct.unpack_from('<i', buf, obj_ptr)[0]
    n = struct.unpack_from('<I', buf, vec_pos)[0]
    results = []
    for i in range(n):
        ep = vec_pos + 4 + i * 4
        op = ep + struct.unpack_from('<i', buf, ep)[0]
        p = rtf(op, 1); begin = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(op, 2); end   = struct.unpack_from('<Q', buf, p)[0] if p else 0
        p = rtf(op, 3); dt    = struct.unpack_from('<B', buf, p)[0] if p else 0
        kvs = []
        ip = rtf(op, 0)
        if ip:
            iv = ip + struct.unpack_from('<i', buf, ip)[0]
            ni = struct.unpack_from('<I', buf, iv)[0]
            for j in range(ni):
                kp = iv + 4 + j*4
                kpos = kp + struct.unpack_from('<i', buf, kp)[0]
                kvs.append(read_kv(buf, kpos))
        results.append({'idx':i,'data_type':dt,'begin':begin,'end':end,'labels':kvs})
    return results

def read_blob(path, begin, end):
    with open(path, 'rb') as f:
        f.seek(begin)
        return f.read(end - begin)

# ── FLATBUFFER BUILDER ────────────────────────────────────────────────
def build_flatbuffer_header(section_infos):
    b = flatbuffers.Builder(65536)
    def make_kv(key, val):
        val_str = b.CreateString(val)
        b.StartObject(1)
        b.PrependUOffsetTRelativeSlot(0, val_str, 0)
        val_table = b.EndObject()
        key_str = b.CreateString(key)
        b.StartObject(3)
        b.PrependUOffsetTRelativeSlot(0, key_str, 0)
        b.PrependByteSlot(1, 9, 0)
        b.PrependUOffsetTRelativeSlot(2, val_table, 0)
        return b.EndObject()

    # Önce tüm section offset'lerini oluştur (NORMAL sırayla)
    section_offsets = []
    for info in section_infos:  # ← reversed() KALDIRILDI
        kv_offsets = [make_kv(k, v) for k, v in reversed(info['labels'])]
        if kv_offsets:
            b.StartVector(4, len(kv_offsets), 4)
            for kv in kv_offsets: b.PrependUOffsetTRelative(kv)
            items_vec = b.EndVector()
        b.StartObject(4)
        if kv_offsets: b.PrependUOffsetTRelativeSlot(0, items_vec, 0)
        b.PrependUint64Slot(1, info['begin'], 0)
        b.PrependUint64Slot(2, info['end'],   0)
        b.PrependByteSlot(3, info['data_type'], 0)
        section_offsets.append(b.EndObject())

    # Vector'a eklerken ters sırayla prepend et (FlatBuffers kuralı)
    b.StartVector(4, len(section_offsets), 4)
    for off in reversed(section_offsets):  # ← burada reversed() KALIR
        b.PrependUOffsetTRelative(off)
    objects_vec = b.EndVector()

    b.StartObject(1)
    b.PrependUOffsetTRelativeSlot(0, objects_vec, 0)
    section_meta = b.EndObject()
    b.StartObject(2)
    b.PrependUOffsetTRelativeSlot(1, section_meta, 0)
    b.Finish(b.EndObject())
    return bytes(b.Output())

def align_to(offset, block=BLOCK_SIZE):
    return (offset + block - 1) // block * block

def compute_layout(fb_size, blobs):
    cursor = align_to(32 + fb_size)
    layout = []
    for blob in blobs:
        begin = cursor; end = begin + len(blob)
        layout.append((begin, end))
        cursor = align_to(end)
    return layout

# ── PARSE ─────────────────────────────────────────────────────────────
print("📖 Modeller parse ediliyor...")
my_sections  = parse_litertlm(MY_PATH)
off_sections = parse_litertlm(OFFICIAL_PATH)

print("\nKendi modelim:")
for s in my_sections:
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {(s['end']-s['begin'])/1e6:>8.1f} MB  {s['labels']}")

# ── BLOB'LARI OKU ────────────────────────────────────────────────────
print("\n📦 Blob'lar okunuyor...")
my_blobs = []
for i, s in enumerate(my_sections):
    if s['data_type'] == 5:  # LlmMetadataProto
        # Resmi modelden al
        blob = read_blob(OFFICIAL_PATH, official_sections[0]['begin'], official_sections[0]['end'])
        print(f"  [{i}] LlmMetadataProto → resmi modelden alındı ({len(blob)} bytes)")
    else:
        blob = read_blob(MY_PATH, s['begin'], s['end'])
    my_blobs.append(blob)

# Resmi modelden: vision_encoder[7], vision_adapter[8], end_of_vision[9]
vision_indices = [7, 8, 9]
vision_blobs   = [read_blob(OFFICIAL_PATH, off_sections[i]['begin'], off_sections[i]['end']) for i in vision_indices]
vision_labels  = [off_sections[i]['labels'] for i in vision_indices]
vision_dtypes  = [off_sections[i]['data_type'] for i in vision_indices]

for i, vi in enumerate(vision_indices):
    size = len(vision_blobs[i]) / 1e6
    print(f"  Resmi [{vi}] → {size:.1f} MB  {vision_labels[i]}")

# ── BİRLEŞTİR ────────────────────────────────────────────────────────
all_blobs  = my_blobs + vision_blobs
all_dtypes = [s['data_type'] for s in my_sections] + vision_dtypes
all_labels = [s['labels']    for s in my_sections] + vision_labels

# İki geçişli layout
dummy_infos = [{'begin':0,'end':0,'data_type':dt,'labels':lb} for dt,lb in zip(all_dtypes, all_labels)]
dummy_fb    = build_flatbuffer_header(dummy_infos)
layout      = compute_layout(len(dummy_fb), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)
layout2     = compute_layout(len(fb_bytes), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout2,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)

print(f"\n✍️  Yazılıyor → {OUTPUT_PATH}")
with open(OUTPUT_PATH, 'wb') as out:
    out.write(b'LITERTLM')
    out.write(struct.pack('<III', 1, 5, 0))
    out.write(b'\x00' * 4)
    out.write(struct.pack('<Q', 32 + len(fb_bytes)))
    out.write(fb_bytes)
    for blob, (begin, end) in zip(all_blobs, layout2):
        cur = out.tell()
        if cur < begin: out.write(b'\x00' * (begin - cur))
        out.write(blob)

total_gb = out.tell() / 1e9 if not out.closed else (layout2[-1][1]) / 1e9
print(f"✅ Tamamlandı: {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB")

# ── DOĞRULA ──────────────────────────────────────────────────────────
print("\n📋 Doğrulama:")
merged_sections = parse_litertlm(OUTPUT_PATH)
for s in merged_sections:
    size = (s['end'] - s['begin']) / 1e6
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {size:>8.1f} MB  {s['labels']}")

📖 Modeller parse ediliyor...

Kendi modelim:
  [0] LlmMetadataProto            0.0 MB  []
  [1] HF_Tokenizer_Zlib           5.5 MB  []
  [2] TFLiteModel              1162.3 MB  [('model_type', 'tf_lite_prefill_decode')]
  [3] TFLiteModel               207.6 MB  [('model_type', 'tf_lite_embedder')]
  [4] TFLiteModel              1180.7 MB  [('model_type', 'tf_lite_per_layer_embedder')]

📦 Blob'lar okunuyor...
  [0] LlmMetadataProto → resmi modelden alındı (12192 bytes)
  Resmi [7] → 219.1 MB  [('model_type', 'tf_lite_vision_encoder')]
  Resmi [8] → 4.7 MB  [('model_type', 'tf_lite_vision_adapter'), ('backend_constraint', 'cpu')]
  Resmi [9] → 0.0 MB  [('model_type', 'tf_lite_end_of_vision')]

✍️  Yazılıyor → /tmp/merged_final.litertlm
✅ Tamamlandı: 2.78 GB

📋 Doğrulama:
  [0] LlmMetadataProto            0.0 MB  []
  [1] HF_Tokenizer_Zlib           5.5 MB  []
  [2] TFLiteModel              1162.3 MB  [('model_type', 'tf_lite_prefill_decode')]
  [3] TFLiteModel               207.6 MB  [('

In [23]:
# LlmMetadataProto'nun gerçekten yazıldığını doğrula
merged = parse_litertlm("/tmp/merged_final.litertlm")
s = merged[0]
actual_size = s['end'] - s['begin']
print(f"Section [0] gerçek boyut: {actual_size} bytes")
print(f"Beklenen: 12192 bytes")
print(f"✅ OK" if actual_size == 12192 else f"❌ HATA — beklenen 12192, gerçek {actual_size}")

Section [0] gerçek boyut: 12192 bytes
Beklenen: 12192 bytes
✅ OK


In [27]:
# Her iki modelden vision encoder blob'larını karşılaştır
with open(OFFICIAL_PATH, 'rb') as f:
    f.seek(official_sections[7]['begin'])
    off_vision = f.read(64)  # ilk 64 byte

with open("/tmp/merged_final.litertlm", 'rb') as f:
    merged = parse_litertlm("/tmp/merged_final.litertlm")
    f.seek(merged[5]['begin'])  # section 5 = vision encoder
    our_vision = f.read(64)

print(f"Resmi vision encoder ilk 64 byte:\n{off_vision.hex()}")
print(f"\nMerge vision encoder ilk 64 byte:\n{our_vision.hex()}")
print(f"\nEşleşiyor mu: {off_vision == our_vision}")

Resmi vision encoder ilk 64 byte:
2000000054464c330000000014002000040008000c0010001400000018001c001400000003000000482700003c270000682900000c0000007c280000bc270000

Merge vision encoder ilk 64 byte:
2000000054464c330000000014002000040008000c0010001400000018001c001400000003000000482700003c270000682900000c0000007c280000bc270000

Eşleşiyor mu: True


In [24]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="/tmp/merged_final.litertlm",
    path_in_repo="model.litertlm",
    repo_id="SalihHub/gemma-4-E2B-it-litert_try",
    repo_type="model",
    commit_message="Fix: use official LlmMetadataProto with vision config (max_num_images fix)",
)
print("✅ Yüklendi!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/merged_final.litertlm  :   6%|5         |  160MB / 2.78GB            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Yüklendi!


In [28]:
import struct, flatbuffers, os

OUTPUT_PATH = "/tmp/merged_official_base.litertlm"
BLOCK_SIZE  = 16 * 1024

# Resmi modelin tüm section'larını parse et
off = parse_litertlm(OFFICIAL_PATH)

# Hangi section'ları alacağız:
# 0  → LlmMetadataProto
# 1  → SP_Tokenizer
# 2  → tf_lite_embedder
# 3  → tf_lite_per_layer_embedder
# 7  → tf_lite_vision_encoder
# 8  → tf_lite_vision_adapter
# 9  → tf_lite_end_of_vision
# 10 → tf_lite_prefill_decode
indices = [0, 1, 2, 3, 7, 8, 9, 10]

all_blobs  = [read_blob(OFFICIAL_PATH, off[i]['begin'], off[i]['end']) for i in indices]
all_dtypes = [off[i]['data_type'] for i in indices]
all_labels = [off[i]['labels']    for i in indices]

print("Section'lar:")
for i, idx in enumerate(indices):
    print(f"  [{i}] {off[idx]['labels']} → {len(all_blobs[i])/1e6:.1f} MB")

# Merge
dummy_infos = [{'begin':0,'end':0,'data_type':dt,'labels':lb} for dt,lb in zip(all_dtypes,all_labels)]
dummy_fb    = build_flatbuffer_header(dummy_infos)
layout      = compute_layout(len(dummy_fb), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)
layout2     = compute_layout(len(fb_bytes), all_blobs)
infos       = [{'begin':b,'end':e,'data_type':dt,'labels':lb} for (b,e),dt,lb in zip(layout2,all_dtypes,all_labels)]
fb_bytes    = build_flatbuffer_header(infos)

with open(OUTPUT_PATH, 'wb') as out:
    out.write(b'LITERTLM')
    out.write(struct.pack('<III', 1, 5, 0))
    out.write(b'\x00' * 4)
    out.write(struct.pack('<Q', 32 + len(fb_bytes)))
    out.write(fb_bytes)
    for blob, (begin, end) in zip(all_blobs, layout2):
        cur = out.tell()
        if cur < begin: out.write(b'\x00' * (begin - cur))
        out.write(blob)

print(f"\n✅ {os.path.getsize(OUTPUT_PATH)/1e9:.2f} GB → {OUTPUT_PATH}")
print("\nDoğrulama:")
for s in parse_litertlm(OUTPUT_PATH):
    size = (s['end']-s['begin'])/1e6
    print(f"  [{s['idx']}] {SECTION_TYPES.get(s['data_type'],'?'):<22} {size:>8.1f} MB  {s['labels']}")

Section'lar:
  [0] [] → 0.0 MB
  [1] [] → 4.7 MB
  [2] [('model_type', 'tf_lite_embedder')] → 103.8 MB
  [3] [('model_type', 'tf_lite_per_layer_embedder')] → 1284.5 MB
  [4] [('model_type', 'tf_lite_vision_encoder')] → 219.1 MB
  [5] [('model_type', 'tf_lite_vision_adapter'), ('backend_constraint', 'cpu')] → 4.7 MB
  [6] [('model_type', 'tf_lite_end_of_vision')] → 0.0 MB
  [7] [('model_type', 'tf_lite_prefill_decode')] → 818.3 MB

✅ 2.44 GB → /tmp/merged_official_base.litertlm

Doğrulama:
  [0] LlmMetadataProto            0.0 MB  []
  [1] SP_Tokenizer                4.7 MB  []
  [2] TFLiteModel               103.8 MB  [('model_type', 'tf_lite_embedder')]
  [3] TFLiteModel              1284.5 MB  [('model_type', 'tf_lite_per_layer_embedder')]
  [4] TFLiteModel               219.1 MB  [('model_type', 'tf_lite_vision_encoder')]
  [5] TFLiteModel                 4.7 MB  [('model_type', 'tf_lite_vision_adapter'), ('backend_constraint', 'cpu')]
  [6] TFLiteModel                 0.0 MB  [('mo

In [30]:
# Resmi modelin decode section'ını incele
import struct

with open(OFFICIAL_PATH, 'rb') as f:
    # Section 10 = tf_lite_prefill_decode
    s = off_sections[10]
    f.seek(s['begin'])
    header = f.read(32)
    print(f"Decode section magic bytes: {header[:8]}")
    print(f"Hex: {header.hex()}")

    # TFLite magic number
    print(f"\nTFLite magic: {header[:4]}")  # TFL3 bekliyoruz

    # Dosya boyutu field'ı (offset 4, 4 byte)
    file_size = struct.unpack_from('<I', header, 4)[0]
    print(f"TFLite dosya boyutu: {file_size / 1e6:.1f} MB")
    print(f"Section gerçek boyutu: {(s['end']-s['begin'])/1e6:.1f} MB")
    print(f"Eşleşiyor mu: {file_size == s['end']-s['begin']}")

Decode section magic bytes: b' \x00\x00\x00TFL3'
Hex: 2000000054464c330000000014002000040008000c0010001400000018001c00

TFLite magic: b' \x00\x00\x00'
TFLite dosya boyutu: 860.6 MB
Section gerçek boyutu: 818.3 MB
Eşleşiyor mu: False


In [31]:
# Resmi modelin tüm section'larını offset sırasıyla listele
print("Resmi model section'ları (offset sırasıyla):")
sorted_sections = sorted(off_sections, key=lambda x: x['begin'])
for s in sorted_sections:
    size = (s['end'] - s['begin']) / 1e6
    print(f"  begin={s['begin']:12d}  end={s['end']:12d}  size={size:8.1f} MB  {s['labels']}")

# Section 10 ve 11 arasında boşluk var mı?
s10 = off_sections[10]
s11 = off_sections[11]
gap = s11['begin'] - s10['end']
print(f"\nSection 10 end:  {s10['end']}")
print(f"Section 11 begin: {s11['begin']}")
print(f"Aradaki boşluk: {gap} bytes")

# TFLite içindeki gerçek dosya boyutu
with open(OFFICIAL_PATH, 'rb') as f:
    f.seek(s10['begin'])
    header = f.read(32)
    # TFLite format: ilk 4 byte padding, sonra "TFL3", sonra 4 byte file_size
    file_size = struct.unpack_from('<I', header, 8)[0]
    print(f"\nTFLite file_size field (offset 8): {file_size / 1e6:.1f} MB")

Resmi model section'ları (offset sırasıyla):
  begin=       16384  end=       28576  size=     0.0 MB  []
  begin=       32768  end=     4721781  size=     4.7 MB  []
  begin=     4734976  end=   108546696  size=   103.8 MB  [('model_type', 'tf_lite_embedder')]
  begin=   108560384  end=  1393078776  size=  1284.5 MB  [('model_type', 'tf_lite_per_layer_embedder')]
  begin=  1393082368  end=  1487134648  size=    94.1 MB  [('model_type', 'tf_lite_audio_encoder_hw'), ('backend_constraint', 'cpu')]
  begin=  1487142912  end=  1496584156  size=     9.4 MB  [('model_type', 'tf_lite_audio_adapter'), ('backend_constraint', 'cpu')]
  begin=  1496596480  end=  1496603252  size=     0.0 MB  [('model_type', 'tf_lite_end_of_audio')]
  begin=  1496612864  end=  1715702984  size=   219.1 MB  [('model_type', 'tf_lite_vision_encoder')]
  begin=  1715716096  end=  1720443208  size=     4.7 MB  [('model_type', 'tf_lite_vision_adapter'), ('backend_constraint', 'cpu')]
  begin=  1720451072  end=  17204578

In [29]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="/tmp/merged_official_base.litertlm",
    path_in_repo="model.litertlm",
    repo_id="SalihHub/gemma-4-E2B-it-litert_try",
    repo_type="model",
    commit_message="Test: resmi model section'ları ile merge (SP_Tokenizer dahil)",
)
print("✅ Yüklendi!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ed_official_base.litertlm:   2%|2         | 55.9MB / 2.44GB            

✅ Yüklendi!
